# Global M5+ earthquakes, 2016–2025

This R notebook explores the documented USGS ComCat snapshot attached to this notebook. The extract contains worldwide earthquake records with preferred magnitude at least 5.0 during the ten complete calendar years 2016–2025. It includes origin time, preferred magnitude and magnitude type, location, depth, and available uncertainty and review fields.

Source: U.S. Geological Survey (USGS), via its [FDSN Earthquake Catalog API](https://earthquake.usgs.gov/fdsnws/event/1/). Credit: U.S. Geological Survey. See the dataset description for source and reuse notes.

## Load the catalog

The notebook works with the Kaggle input path and with a local copy of the CSV.

In [ ]:
input_files <- list.files('/kaggle/input', pattern = '^earthquakes_m5_2016_2025\\.csv$',
                          recursive = TRUE, full.names = TRUE)
if (length(input_files) == 0) {
  input_files <- 'earthquakes_m5_2016_2025.csv'
}
if (!file.exists(input_files[1])) {
  stop('Catalog CSV not found. Attach the USGS M5+ dataset or run from its local folder.')
}

events <- read.csv(input_files[1], stringsAsFactors = FALSE, na.strings = c('', 'NA'))
events$time <- as.POSIXct(sub('Z$', '', events$time),
                          format = '%Y-%m-%dT%H:%M:%OS', tz = 'UTC')
events$year <- as.integer(format(events$time, '%Y', tz = 'UTC'))

cat('Rows:', nrow(events), '\n')
cat('Unique event IDs:', length(unique(events$id)), '\n')
cat('Time span (UTC):', format(min(events$time), '%Y-%m-%d'), 'to',
    format(max(events$time), '%Y-%m-%d'), '\n')
cat('Magnitude range:', min(events$mag, na.rm = TRUE), 'to',
    max(events$mag, na.rm = TRUE), '\n')

## Annual catalog counts

These are counts of records meeting this notebook's M5+ threshold. They can change with detection, network coverage, and catalog review; they are not estimates of the Earth's overall earthquake rate.

In [ ]:
annual_counts <- table(factor(events$year, levels = 2016:2025))
barplot(annual_counts,
        main = 'USGS ComCat records with magnitude ≥ 5.0',
        xlab = 'Origin year (UTC)', ylab = 'Catalog records',
        col = '#3B82A0', border = NA, las = 2)

## Preferred magnitude distribution

Magnitude types are not all the same scale. The histogram is useful for exploring this extract, while magType should be retained for analyses that compare magnitude measurements.

In [ ]:
hist(events$mag, breaks = 30,
     main = 'Preferred magnitudes in the M5+ extract',
     xlab = 'Preferred magnitude', ylab = 'Catalog records',
     col = '#E5A44A', border = 'white')

head(sort(table(events$magType), decreasing = TRUE), 10)

## Depth and magnitude

A transparent point sample is shown at low opacity to keep dense areas visible. Depth is the catalog's preferred depth in kilometers.

In [ ]:
plot(events$depth, events$mag,
     pch = 16, cex = 0.35,
     col = grDevices::rgb(0.08, 0.35, 0.55, alpha = 0.22),
     main = 'Preferred depth and magnitude',
     xlab = 'Depth (km)', ylab = 'Preferred magnitude')
abline(h = 5, col = '#B45309', lty = 2)

## Largest preferred magnitudes in this snapshot

In [ ]:
strongest <- events[order(events$mag, decreasing = TRUE),
                    c('time', 'mag', 'magType', 'place', 'depth', 'id')]
head(strongest, 15)

## Global epicenter overview

A simple equirectangular view plots the coordinates directly from the catalog. It has no basemap, so the grid is there to help locate longitude and latitude rather than imply country boundaries.

In [ ]:
if (dir.exists('/kaggle/working')) {
  map_file <- '/kaggle/working/usgs-m5-epicenter-map.png'
  png(map_file, width = 1600, height = 900, res = 150, bg = '#101827')
  par(mar = c(6.5, 4.5, 4.5, 1.5), bg = '#101827', fg = '#E6EDF5',
      col.axis = '#CBD5E1', col.lab = '#E6EDF5',
      col.main = '#F8FAFC', cex.main = 1.35)
  plot(events$longitude, events$latitude, type = 'n',
       xlim = c(-180, 180), ylim = c(-90, 90), axes = FALSE,
       xlab = 'Longitude', ylab = 'Latitude',
       main = paste(format(nrow(events), big.mark = ','),
                    'USGS M5+ epicenters | 2016–2025'))
  abline(v = seq(-180, 180, by = 60), h = seq(-90, 90, by = 30),
         col = '#26364D', lty = 1)
  axis(1, at = seq(-180, 180, by = 60),
       labels = c('180°W', '120°W', '60°W', '0°', '60°E', '120°E', '180°E'))
  axis(2, at = seq(-90, 90, by = 30),
       labels = c('90°S', '60°S', '30°S', '0°', '30°N', '60°N', '90°N'), las = 1)
  points(events$longitude, events$latitude, pch = 16, cex = 0.42,
         col = grDevices::rgb(0.30, 0.72, 0.94, alpha = 0.30))
  box(col = '#718096')
  mtext('USGS ComCat preferred epicenters · Data snapshot and source notes in the dataset',
        side = 1, line = 4.7, col = '#B9C6D6', cex = 0.82)
  dev.off()
  cat('Map saved to', map_file, '\n')
}

## Limits and source

The catalog is a thresholded, revisable source. Events below magnitude 5.0 are absent, preferred solutions can be updated, and reporting varies across time and contributing networks. Do not interpret the annual bars as a direct trend in global seismicity. Preserve the USGS event ID and source credit when using records. See the dataset README and [USGS catalog API documentation](https://earthquake.usgs.gov/fdsnws/event/1/) for the exact query, fields, and reuse notes.